# 02 — Feature Exploration

**Goal:** explore the engineered RFM features and validate the churn label definition.

Topics covered:
- RFM distributions
- Churn rate by RFM segment
- Correlation heatmap
- Feature vs churn label plots

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.express as px

import config
from src.features import build_rfm_features, add_churn_label, FEATURE_COLS

In [2]:
rfm = build_rfm_features()
df  = add_churn_label(rfm)

print(f"Customers: {len(df):,}")
print(f"Churn rate: {df['churned'].mean():.1%}")
df[FEATURE_COLS + ['churned']].describe()

2026-04-25 17:04:11 | INFO     | src.features — Built RFM features for 93357 customers
2026-04-25 17:04:11 | INFO     | src.features — Churn label applied — rate: 59.3% (180-day window)


Customers: 93,357
Churn rate: 59.3%


,recency_days,frequency,monetary_value,customer_age_days,churned
count,93357.000000,93357.000000,93357.000000,93357.000000,93357.000000
mean,237.473783,1.023887,165.198772,240.118898,0.593207
std,152.587935,0.177241,226.314579,153.095803,0.491238
min,0.000000,1.000000,9.590000,0.000000,0.000000
25%,114.000000,1.000000,63.060000,116.000000,0.000000
50%,218.000000,1.000000,107.780000,221.000000,1.000000
75%,346.000000,1.000000,182.560000,350.000000,1.000000
max,695.000000,15.000000,13664.080000,695.000000,1.000000


In [3]:
# --- Recency distribution by churn label ---
px.histogram(df, x="recency_days", color="churned",
             barmode="overlay", nbins=60,
             title="Recency by Churn Status",
             labels={"recency_days": "Days Since Last Purchase", "churned": "Churned"})

In [4]:
# --- Frequency vs Monetary scatter ---
px.scatter(df.sample(5000, random_state=42), x="frequency", y="monetary_value",
           color="churned", opacity=0.5, log_y=True,
           title="Frequency vs Monetary (log scale)",
           labels={"frequency": "Order Count", "monetary_value": "Total Revenue (R$)"})

In [5]:
# --- Correlation heatmap ---
corr = df[FEATURE_COLS + ['churned']].corr()
px.imshow(corr, text_auto=True, title="Feature Correlation Matrix",
          color_continuous_scale="RdBu_r", zmin=-1, zmax=1)